### Plot the resampled classification results

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
data = pd.read_csv("classification_results.csv")


In [ ]:
g = sns.catplot(
    data=data, kind="bar",
    x="Scheme", y="F1_score", hue="Sensor", 
    palette="colorblind", alpha=1, height=6, aspect = 2
)
g.despine(left=True)
g.set(ylim = (0.5, 1))
g.set_axis_labels("Classification scheme", " F1 score")
g.legend.set_title("")
g.savefig(r"C:\Users\s4770224\Documents\Work\Writing\Figures\Obj1\part2\class_sensor_scores.svg")

In [5]:
resamp_dir = "data/processed/resampled/"
scheme_list = ["Class", "ko", "bo", "brgm", "kbrgm"] # "kelps-loc", "browns-loc"]

In [ ]:
# Import labels
labels = pd.read_csv("data/labels_prepped.csv", index_col=0)

# Make new desired labels
labels['bo'] = labels['kbom'].replace(["other", "mineral"], "other")
labels['bo'] = labels['bo'].replace(["kelp", "other_brown_alg"], "brown_algae")
labels['brgm'] = labels['kbrgm'].replace(["kelp", "other_brown_alg"], "brown_algae")
labels["genus-loc"] = labels["Class"] + "-" + labels["site"]

labels.head()

In [7]:
spectra = {}
for file in glob.glob(os.path.join(resamp_dir, "noisy_*.csv")):
    #remove the .csv extension
    filename = os.path.basename(file)[:-4]
    # read in the data
    spectra[filename] = pd.read_csv(file, index_col=0).dropna(axis = 1, how = "all")
del spectra["noisy_Dove_resampled"]

In [8]:
#Change band names to central wavelengths for plotting
central_nm = {
    "noisy_Enmap_resampled" : [459.03, 463.73, 468.41, 473.08, 477.74, 482.41, 487.09, 491.78, 496.5, 501.24, 506.02, 510.83, 515.67, 520.55,
    525.47, 530.42, 535.42, 540.46, 545.55, 550.69, 555.87,
    561.11, 566.4, 571.76, 577.17, 582.64, 588.17, 593.77,
    599.45, 605.19, 611.02, 616.92, 622.92, 628.99, 635.11,
    641.29, 647.54, 653.84, 660.21, 666.64, 673.13, 679.69,
    686.32, 693.01, 699.78, 706.62, 713.52, 720.5, 727.54,
    734.65, 741.83, 749.06, 756.35, 763.7, 771.11, 778.57,
    786.08, 793.64, 801.25, 808.9, 816.61, 824.36, 832.14,
    839.98, 847.85, 855.76, 863.7, 871.68, 879.69, 887.73,
    895.79, 901.96, 903.87, 911.57, 911.97, 920.08, 921.32,
    928.2],
    
    'noisy_Landsat_9_resampled' : [481.89, 560.95, 654.32, 864.64, 925],
    'noisy_Sentinel_2_ABC_resampled' :[492.0, 560.5, 665.0, 705.0,741.0, 784.0, 842.0, 865.0],
    'noisy_Skysat_resampled': [483 , 555, 650, 820, 925],
    'noisy_Superdove_resampled': [490, 531 , 565, 610, 665, 705, 865],
}

for item in spectra.items():
    item[1].columns = central_nm[item[0]]

In [ ]:
fig, ax = plt.subplots(len(scheme_list), len(spectra.items()), figsize = (10,10), sharey = True, sharex = "col")
fig.tight_layout(pad = 2)

i = 0
j = 0
for key in spectra.keys():
    data = spectra[key]
    for scheme in scheme_list:
        ax[i,j].plot(data.groupby(labels[scheme]).mean().T, label = labels[scheme].unique())
        ax[i,j].set_title(f"{key[6: -10]} - {scheme}", fontsize = "small")
        ax[i,j].legend(loc = "upper right", fontsize = "x-small")
        ax[i,j].xaxis.set_ticks(np.arange(450, 950, 100))
        i+= 1
        
    i=0
    j+=1

plt.savefig(r"C:\Users\s4770224\Documents\Work\Writing\Figures\Obj1\part2\class_scheme_spectra.svg")

In [19]:
rsr = pd.read_csv("satellite_RSRs/Enmap.csv", index_col = 0)

In [ ]:
plt.plot(rsr)
plt.xlim(450, 950)
plt.savefig(r"C:\Users\s4770224\Documents\Work\Writing\Figures\Obj1\part2\enmap_rsr.svg")